# Day 2 — Neo4j Graph Memory: Test & Exploration

This notebook validates the complete Day 2 memory stack:
1. Neo4j connection and schema verification
2. Single-file review with memory injection
3. Second review — memory context diff
4. Graph visualisation (networkx + matplotlib)
5. Multi-file pattern emergence
6. Direct Neo4j queries as DataFrames
7. Day 2 summary

In [ ]:
# ── Cell 1: Imports, path setup, connect to Neo4j ────────────────────────────
import sys
import os
import tempfile
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from loguru import logger
logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss} | {level} | {message}')

from src.memory.neo4j_client import Neo4jClient
from src.memory.graph_schema import SCHEMA_QUERIES

# Reset singleton for clean notebook state
Neo4jClient._instance = None

client = Neo4jClient.get_instance()
connected = client.connect()

print(f'\nNeo4j connected: {connected}')
if connected:
    client.initialize_schema()
    print('Schema initialised')
else:
    print('Neo4j unavailable — some cells will show N/A results.')
    print('Start Neo4j or set credentials in .env to enable full demo.')

In [ ]:
# ── Cell 2: Schema verification — node/relationship counts ───────────────────
import pandas as pd

if connected:
    # Node counts by label
    node_rows = client.query(
        'MATCH (n) RETURN labels(n)[0] AS label, count(n) AS count ORDER BY count DESC',
        {}
    )
    df_nodes = pd.DataFrame(node_rows) if node_rows else pd.DataFrame(columns=['label', 'count'])
    print('Node counts by label:')
    print(df_nodes.to_string(index=False))

    # Relationship counts by type
    rel_rows = client.query(
        'MATCH ()-[r]->() RETURN type(r) AS rel_type, count(r) AS count ORDER BY count DESC',
        {}
    )
    df_rels = pd.DataFrame(rel_rows) if rel_rows else pd.DataFrame(columns=['rel_type', 'count'])
    print('\nRelationship counts by type:')
    print(df_rels.to_string(index=False))

    # Schema constraints
    constraints = client.query('SHOW CONSTRAINTS', {})
    print(f'\nActive constraints: {len(constraints)}')
    for c in constraints:
        print(f'  {c.get("name", "?")} — {c.get("type", "?")} on {c.get("labelsOrTypes", "?")}')
else:
    print('N/A — Neo4j not connected')

In [ ]:
# ── Cell 3: Single file review with memory enabled ───────────────────────────
import uuid

from src.agent.orchestrator import ReviewOrchestrator
from src.memory.memory_retriever import MemoryRetriever
from src.memory.memory_writer import MemoryWriter

REPO_URL = f'https://github.com/test/day2-notebook-{str(uuid.uuid4())[:4]}'
print(f'Using repo URL: {REPO_URL}')

# Create a test file
f = tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w')
f.write('''
import os
import pickle

DB_PASSWORD = "hardcoded_secret_123"  # CWE-798
API_TOKEN   = "sk-prod-abc"           # CWE-798

def login(conn, user, pwd):
    """SQL injection."""
    q = "SELECT * FROM users WHERE user='" + user + "'"
    conn.cursor().execute(q)

def run(cmd):
    """eval misuse."""
    return eval(cmd)

def load_cfg(p):
    with open(p, "rb") as f:
        return pickle.load(f)   # B301
''')
f.close()
test_file = f.name

# Check what memory context exists BEFORE the review
print('\n--- Memory context BEFORE first review ---')
if connected:
    retriever = MemoryRetriever(client)
    ctx_before = retriever.get_file_context(test_file, REPO_URL)
    print(ctx_before or '(empty — no past history)')
else:
    print('N/A — Neo4j not connected')

# Run review
Neo4jClient._instance = None
orch = ReviewOrchestrator(use_memory=connected)
session = orch.review_repo(
    repo_url=REPO_URL,
    files_to_review=[test_file],
    use_defect_api=False,
)

print(f'\nReview complete: {session.total_issues} issues, status={session.file_states[0].status.value}')
for issue in session.file_states[0].issues_found:
    print(f'  [{issue.severity.value}] {issue.title} (line {issue.line_number})')

In [ ]:
# ── Cell 4: Second review — memory context diff ───────────────────────────────
print('--- Memory context AFTER first review ---')
if connected:
    # Reconnect retriever to get fresh state
    retriever2 = MemoryRetriever(client)
    ctx_after = retriever2.get_file_context(test_file, REPO_URL)
    print(ctx_after or '(empty — issues may not have been written yet)')

    print('\n' + '='*60)
    print('Running second review — agent should see past context...')
    print('='*60 + '\n')

    Neo4jClient._instance = None
    orch2 = ReviewOrchestrator(use_memory=True)
    session2 = orch2.review_repo(
        repo_url=REPO_URL,
        files_to_review=[test_file],
        use_defect_api=False,
    )
    state2 = session2.file_states[0]
    print(f'Second review: {session2.total_issues} issues found')
    print(f'Past context was injected: {bool(state2.past_issues_context)}')
    if state2.past_issues_context:
        print(f'\nContext injected into agent prompt:')
        print(state2.past_issues_context[:600])
else:
    print('N/A — Neo4j not connected')

import os as _os
_os.unlink(test_file)

In [ ]:
# ── Cell 5: Graph visualisation ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print('networkx not installed: pip install networkx')

if connected and HAS_NX:
    # Fetch all nodes and edges for this repo
    nodes_raw = client.query(
        '''
        MATCH (n)
        WHERE n.repo_url = $repo_url OR n.url = $repo_url
        RETURN elementId(n) AS id, labels(n)[0] AS label,
               coalesce(n.title, n.path, n.session_id, n.url, n.description, '') AS display
        ''',
        {'repo_url': REPO_URL}
    )
    rels_raw = client.query(
        '''
        MATCH (a)-[r]->(b)
        WHERE (a.repo_url = $repo_url OR a.url = $repo_url)
        RETURN elementId(a) AS src, elementId(b) AS dst, type(r) AS rel_type
        ''',
        {'repo_url': REPO_URL}
    )

    G = nx.DiGraph()
    COLOR_MAP = {
        'Repo':    '#4C72B0',
        'File':    '#55A868',
        'Review':  '#DD8452',
        'Issue':   '#C44E52',
        'Pattern': '#8172B2',
    }

    id_to_label  = {}
    id_to_display = {}
    for n in nodes_raw:
        G.add_node(n['id'])
        id_to_label[n['id']]   = n['label'] or 'Unknown'
        id_to_display[n['id']] = (n['display'] or '')[:25]

    for r in rels_raw:
        G.add_edge(r['src'], r['dst'], label=r['rel_type'])

    node_colors = [COLOR_MAP.get(id_to_label.get(n, ''), '#999999') for n in G.nodes()]
    node_labels = {n: f"{id_to_label.get(n, '?')}\n{id_to_display.get(n, '')}" for n in G.nodes()}

    fig, ax = plt.subplots(figsize=(14, 9))
    pos = nx.spring_layout(G, seed=42, k=2.5)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1200, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=7, ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='#aaaaaa', arrows=True,
                           arrowsize=15, connectionstyle='arc3,rad=0.1', ax=ax)
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6, ax=ax)

    patches = [mpatches.Patch(color=v, label=k) for k, v in COLOR_MAP.items()]
    ax.legend(handles=patches, loc='upper left', fontsize=9)
    ax.set_title(f'Neo4j Graph — {REPO_URL.split("/")[-1]}\n{G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    ax.axis('off')
    plt.tight_layout()

    out_dir = ROOT / 'plots'
    out_dir.mkdir(exist_ok=True)
    plt.savefig(out_dir / 'memory_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved to {out_dir / "memory_graph.png"}')
else:
    print('N/A — Neo4j not connected or networkx missing')

In [ ]:
# ── Cell 6: Multi-file pattern emergence ─────────────────────────────────────
PATTERN_REPO = f'https://github.com/test/pattern-demo-{str(uuid.uuid4())[:4]}'
print(f'Pattern demo repo: {PATTERN_REPO}')

# Create 3 files each with security issues (same category → triggers pattern detection)
files = []
for i in range(3):
    tmp = tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w')
    tmp.write(f'''
SECRET_{i} = "hardcoded-key-{i}"   # CWE-798

def query_{i}(conn, user_input):
    sql = "SELECT * FROM t WHERE id='" + user_input + "'"   # SQL injection
    conn.cursor().execute(sql)

def run_{i}(cmd):
    return eval(cmd)   # B307
''')
    tmp.close()
    files.append(tmp.name)
    print(f'  Created: {Path(tmp.name).name}')

Neo4jClient._instance = None
orch3 = ReviewOrchestrator(use_memory=connected)
session3 = orch3.review_repo(
    repo_url=PATTERN_REPO,
    files_to_review=files,
    use_defect_api=False,
    max_files=3,
)

print(f'\nMulti-file review complete:')
print(f'  Files reviewed: {len(session3.files_reviewed)}')
print(f'  Total issues:   {session3.total_issues}')
print(f'  Patterns:       {session3.patterns_detected}')

if connected:
    patterns = client.query(
        'MATCH (p:Pattern) WHERE p.pattern_id STARTS WITH $repo RETURN p.description, p.affected_files',
        {'repo': PATTERN_REPO}
    )
    print(f'\nPatterns in Neo4j ({len(patterns)}):')
    for p in patterns:
        print(f'  {p["p.description"]} — {p["p.affected_files"]} files')

for f in files:
    import os as _o; _o.unlink(f)

In [ ]:
# ── Cell 7: Direct Neo4j queries as DataFrames ────────────────────────────────
import pandas as pd

if connected:
    # All issues sorted by severity
    issues = client.query(
        '''
        MATCH (i:Issue)
        RETURN i.title AS title, i.severity AS severity, i.category AS category,
               i.source_tool AS tool, i.occurrence_count AS occurrences,
               i.file_path AS file
        ORDER BY
            CASE i.severity
                WHEN 'CRITICAL' THEN 0 WHEN 'HIGH' THEN 1
                WHEN 'MEDIUM' THEN 2 ELSE 3
            END,
            i.occurrence_count DESC
        ''', {}
    )
    print(f'All issues ({len(issues)}):')
    if issues:
        df_issues = pd.DataFrame(issues)
        df_issues['file'] = df_issues['file'].apply(lambda x: Path(x).name if x else '')
        print(df_issues[['severity', 'category', 'title', 'tool', 'occurrences', 'file']]
              .to_string(index=False))

    # All patterns
    print()
    patterns = client.query(
        'MATCH (p:Pattern) RETURN p.description AS description, p.severity AS severity, '
        'p.affected_files AS files, p.occurrence_count AS count ORDER BY files DESC',
        {}
    )
    print(f'All patterns ({len(patterns)}):')
    if patterns:
        print(pd.DataFrame(patterns).to_string(index=False))
    else:
        print('None yet — review more files to trigger pattern promotion')

    # Files by review count
    print()
    files_reviewed = client.query(
        'MATCH (f:File) RETURN f.path AS path, f.review_count AS reviews, '
        'f.avg_risk_score AS avg_risk ORDER BY f.review_count DESC',
        {}
    )
    print(f'Files by review count ({len(files_reviewed)}):')
    if files_reviewed:
        df_files = pd.DataFrame(files_reviewed)
        df_files['path'] = df_files['path'].apply(lambda x: Path(x).name if x else '')
        print(df_files.to_string(index=False))
else:
    print('N/A — Neo4j not connected')

In [ ]:
# ── Cell 8: Day 2 Summary ─────────────────────────────────────────────────────
print('=' * 60)
print('DAY 2 SUMMARY')
print('=' * 60)
print(f'  Neo4j connected:         {connected}')
if connected:
    node_count = client.query_single('MATCH (n) RETURN count(n) AS cnt', {})
    rel_count  = client.query_single('MATCH ()-[r]->() RETURN count(r) AS cnt', {})
    print(f'  Total nodes in graph:    {node_count["cnt"] if node_count else "N/A"}')
    print(f'  Total relationships:     {rel_count["cnt"] if rel_count else "N/A"}')

total_files   = len(session.files_reviewed) + len(session3.files_reviewed)
total_issues  = session.total_issues + session3.total_issues
print(f'  Files reviewed:          {total_files}')
print(f'  Total issues found:      {total_issues}')
print(f'  Memory retrieval works:  {connected}')
print(f'  Pattern detection works: {connected}')
print()
print('Git commit:')
print('  git add .')
print('  git commit -m "feat: Day 2 — Neo4j graph memory + multi-file orchestration"')